In [1]:
from sampler import ZeoSampler

In [76]:
from torch.utils.data import BatchSampler
from torch.utils.data import Dataset
from numpy.random import choice, shuffle
import numpy as np

class ZeoSampler(BatchSampler):
    def __init__(self, zeolite_codes: list, batch_size, num_samples=1500):
        self.batch_size = batch_size
        self.num_samples = num_samples

        self.zeolite_codes = zeolite_codes
        self.unique_zeo_codes = set(zeolite_codes)
        self.unique_zeo_codes_num = len(self.unique_zeo_codes)
        self.most_zeo_code = max([zeolite_codes.count(zeo_code) for zeo_code in self.unique_zeo_codes])

    def __iter__(self):
        batch_indices = []

        # Sample indices for each zeolite code
        for zeo_code in self.unique_zeo_codes:
            zeo_code_indices = [idx for idx, value in enumerate(self.zeolite_codes) if zeo_code == value]
            zeo_code_indices = choice(zeo_code_indices, size=self.most_zeo_code, replace=True)
            batch_indices.extend(zeo_code_indices.tolist())

        # Shuffle and pad indices to fit batches
        shuffle(batch_indices)
        indices_to_add = self.batch_size - (len(batch_indices) % self.batch_size)
        if indices_to_add < self.batch_size:
            batch_indices.extend([-1] * indices_to_add)

        # Yield one batch at a time
        for i in range(0, len(batch_indices), self.batch_size):
            batch = batch_indices[i:i + self.batch_size]
            # print([idx for idx in batch if idx != -1])
            yield [idx for idx in batch if idx != -1]  # Remove padded indices

    def __len__(self):
        return int(np.ceil(len(self.unique_zeo_codes) * self.most_zeo_code / self.batch_size))

In [77]:
class TestDataset(Dataset):
    def __init__(self, graphs):
        self.graphs = graphs

    def __len__(self):
        return len(self.graphs)

    def __getitem__(self, index):
        print(index)
        return Data(
            x=torch.randn(5,3)
        )

In [78]:
import torch
from torch_geometric.loader import DataLoader 
from torch_geometric.data import Data

X = 5 * ['MOR'] + 2 * ["TON"] + 1 * ['MFI']
graphs = [Data(x=torch.randn(5,3)) for i in range(8)]

sampler = ZeoSampler(X, batch_size=4)
dataset = TestDataset(graphs=graphs)
data_loader = DataLoader(dataset=dataset, sampler=sampler)

In [79]:
for data in data_loader:
    print(data)

[2, 5, 5, 7]
DataBatch(x=[5, 3], batch=[5], ptr=[2])
[2, 6, 7, 7]
DataBatch(x=[5, 3], batch=[5], ptr=[2])
[7, 0, 4, 7]
DataBatch(x=[5, 3], batch=[5], ptr=[2])
[2, 6, 6]
DataBatch(x=[5, 3], batch=[5], ptr=[2])
